# 🏦 Crédit Bancaire & Scoring — Prédiction du Montant du Prêt (en Dirhams)

**Objectif :** Prédire le montant du prêt accordé (`LoanAmount`) à partir de variables financières et socio-économiques.  
**Modèles :** Régression Linéaire, Ridge (L2), Lasso (L1), ElasticNet  
**Dataset :** Loan Prediction Dataset (Kaggle) — adapté au contexte marocain (Bank Al-Maghrib)

---
### 📌 Plan du notebook
1. Import des bibliothèques
2. Chargement et exploration des données
3. Nettoyage et prétraitement
4. Analyse exploratoire (EDA)
5. Encodage et feature engineering
6. Entraînement des modèles (Ridge / Lasso / ElasticNet)
7. Évaluation et comparaison des modèles
8. Interprétation des coefficients
9. Prédiction sur un nouveau client

## 1. 📦 Import des bibliothèques

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, RidgeCV, LassoCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

import matplotlib
matplotlib.rcParams['figure.figsize'] = (12, 6)
matplotlib.rcParams['font.size'] = 12
sns.set_style('whitegrid')
sns.set_palette('husl')

print('✅ Bibliothèques importées avec succès')

## 2. 📂 Chargement des données

In [ ]:
# ─── Option 1 : Charger depuis Google Drive ───
# from google.colab import drive
# drive.mount('/content/drive')
# df = pd.read_csv('/content/drive/MyDrive/loan_data_set.csv', sep=';')

# ─── Option 2 : Upload direct dans Colab ───
# from google.colab import files
# uploaded = files.upload()
# df = pd.read_csv(list(uploaded.keys())[0], sep=';')

# ─── Option 3 : Depuis URL Kaggle ───
# !pip install kaggle
# ... ou directement depuis un lien public

# ─── DÉMONSTRATION : Données synthétiques réalistes (structure identique) ───
# Remplacez ce bloc par l'une des options ci-dessus avec votre fichier

np.random.seed(42)
n = 614

df = pd.DataFrame({
    'Loan_ID': [f'LP{100000+i}' for i in range(n)],
    'Gender': np.random.choice(['Male', 'Female'], n, p=[0.81, 0.19]),
    'Married': np.random.choice(['Yes', 'No'], n, p=[0.65, 0.35]),
    'Dependents': np.random.choice(['0', '1', '2', '3+'], n, p=[0.57, 0.16, 0.17, 0.10]),
    'Education': np.random.choice(['Graduate', 'Not Graduate'], n, p=[0.78, 0.22]),
    'Self_Employed': np.random.choice(['No', 'Yes'], n, p=[0.86, 0.14]),
    'ApplicantIncome': np.random.lognormal(8.4, 0.6, n).astype(int),
    'CoapplicantIncome': np.where(np.random.rand(n) > 0.4,
                                   np.random.lognormal(7.0, 0.7, n), 0).astype(float),
    'LoanAmount': np.random.lognormal(4.8, 0.5, n),
    'Loan_Amount_Term': np.random.choice([120, 180, 240, 300, 360, 480], n, p=[0.03,0.04,0.05,0.06,0.79,0.03]),
    'Credit_History': np.random.choice([1.0, 0.0, np.nan], n, p=[0.78, 0.16, 0.06]),
    'Property_Area': np.random.choice(['Urban', 'Semiurban', 'Rural'], n, p=[0.38,0.37,0.25]),
    'Loan_Status': np.random.choice(['Y', 'N'], n, p=[0.69, 0.31])
})

# Conversion en Dirhams (1 USD ≈ 10 MAD, 1 INR ≈ 0.12 MAD)
df['LoanAmount_MAD'] = (df['LoanAmount'] * 1000 * 0.12).round(0)  # en dirhams
df['ApplicantIncome_MAD'] = (df['ApplicantIncome'] * 0.12).round(0)
df['CoapplicantIncome_MAD'] = (df['CoapplicantIncome'] * 0.12).round(0)

print(f'✅ Dataset chargé : {df.shape[0]} lignes × {df.shape[1]} colonnes')
df.head()

In [ ]:
# ─── Aperçu général ───
print('=== INFORMATIONS GÉNÉRALES ===')
df.info()
print()
print('=== STATISTIQUES DESCRIPTIVES ===')
df.describe()

In [ ]:
# ─── Valeurs manquantes ───
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Manquants': missing, '% Manquant': missing_pct})
missing_df = missing_df[missing_df['Manquants'] > 0].sort_values('% Manquant', ascending=False)

print('=== VALEURS MANQUANTES ===')
print(missing_df)

## 3. 🧹 Nettoyage et Prétraitement

In [ ]:
df_clean = df.copy()

# ─── 1. Supprimer la colonne ID ───
df_clean.drop(columns=['Loan_ID'], inplace=True)

# ─── 2. Variables catégorielles : remplacement par le mode ───
cat_cols = ['Gender', 'Married', 'Dependents', 'Self_Employed', 'Loan_Status']
for col in cat_cols:
    df_clean[col].fillna(df_clean[col].mode()[0], inplace=True)

# ─── 3. Credit_History : valeur médiane ───
df_clean['Credit_History'].fillna(df_clean['Credit_History'].median(), inplace=True)

# ─── 4. LoanAmount : médiane ───
df_clean['LoanAmount'].fillna(df_clean['LoanAmount'].median(), inplace=True)
df_clean['LoanAmount_MAD'].fillna(df_clean['LoanAmount_MAD'].median(), inplace=True)

# ─── 5. Loan_Amount_Term : mode ───
df_clean['Loan_Amount_Term'].fillna(df_clean['Loan_Amount_Term'].mode()[0], inplace=True)

print(f'✅ Valeurs manquantes restantes : {df_clean.isnull().sum().sum()}')
print(f'   Shape final : {df_clean.shape}')

## 4. 📊 Analyse Exploratoire (EDA)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Distribution du montant du prêt en MAD
axes[0].hist(df_clean['LoanAmount_MAD'], bins=40, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].set_title('Distribution du Montant du Prêt (MAD)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Montant (Dirhams)')
axes[0].set_ylabel('Fréquence')

# Log-transformation
axes[1].hist(np.log1p(df_clean['LoanAmount_MAD']), bins=40, color='coral', edgecolor='white', alpha=0.8)
axes[1].set_title('Log(Montant du Prêt + 1)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('log(Montant)')

# Revenu vs Montant
axes[2].scatter(df_clean['ApplicantIncome_MAD'], df_clean['LoanAmount_MAD'],
                alpha=0.4, color='green', s=20)
axes[2].set_title('Revenu Demandeur vs Montant Prêt', fontsize=13, fontweight='bold')
axes[2].set_xlabel('Revenu (MAD)')
axes[2].set_ylabel('Montant Prêt (MAD)')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Montant par Education
df_clean.groupby('Education')['LoanAmount_MAD'].median().plot(kind='bar', ax=axes[0],
    color=['#2ecc71', '#e74c3c'], edgecolor='white')
axes[0].set_title('Montant Médian par Niveau d\'Éducation', fontweight='bold')
axes[0].set_xlabel('')
axes[0].set_ylabel('Montant médian (MAD)')
axes[0].tick_params(axis='x', rotation=0)

# Montant par Property Area
df_clean.groupby('Property_Area')['LoanAmount_MAD'].median().plot(kind='bar', ax=axes[1],
    color=['#3498db', '#9b59b6', '#f39c12'], edgecolor='white')
axes[1].set_title('Montant Médian par Zone Géographique', fontweight='bold')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=0)

# Credit History
df_clean.groupby('Credit_History')['LoanAmount_MAD'].median().plot(kind='bar', ax=axes[2],
    color=['#e74c3c', '#27ae60'], edgecolor='white')
axes[2].set_title('Montant Médian par Historique de Crédit', fontweight='bold')
axes[2].set_xlabel('0 = Mauvais | 1 = Bon')
axes[2].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# ─── Matrice de corrélation ───
num_cols = ['ApplicantIncome_MAD', 'CoapplicantIncome_MAD', 'LoanAmount_MAD',
            'Loan_Amount_Term', 'Credit_History']
corr_matrix = df_clean[num_cols].corr()

plt.figure(figsize=(9, 7))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdYlGn',
            mask=mask, vmin=-1, vmax=1,
            linewidths=0.5, square=True, cbar_kws={'shrink': 0.8})
plt.title('Matrice de Corrélation — Variables Numériques', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. 🔧 Feature Engineering & Encodage

In [ ]:
df_model = df_clean.copy()

# ─── 1. Feature Engineering ───
df_model['TotalIncome_MAD'] = df_model['ApplicantIncome_MAD'] + df_model['CoapplicantIncome_MAD']
df_model['DebtToIncome'] = df_model['LoanAmount_MAD'] / (df_model['TotalIncome_MAD'] + 1)
df_model['IncomePerTerm'] = df_model['TotalIncome_MAD'] / (df_model['Loan_Amount_Term'] + 1)
df_model['LogLoanAmount'] = np.log1p(df_model['LoanAmount_MAD'])   # variable cible transformée
df_model['LogIncome'] = np.log1p(df_model['TotalIncome_MAD'])

# ─── 2. Encodage des variables catégorielles ───
binary_maps = {
    'Gender': {'Male': 1, 'Female': 0},
    'Married': {'Yes': 1, 'No': 0},
    'Education': {'Graduate': 1, 'Not Graduate': 0},
    'Self_Employed': {'Yes': 1, 'No': 0},
    'Loan_Status': {'Y': 1, 'N': 0},
}
for col, mapping in binary_maps.items():
    df_model[col] = df_model[col].map(mapping)

# Dependents
df_model['Dependents'] = df_model['Dependents'].replace({'3+': 3}).astype(float)

# One-Hot Encoding pour Property_Area
df_model = pd.get_dummies(df_model, columns=['Property_Area'], drop_first=True)

print('✅ Nouvelles colonnes créées :')
print(df_model.columns.tolist())
df_model.head(3)

In [ ]:
# ─── Sélection des features et de la cible ───
FEATURES = [
    'Gender', 'Married', 'Dependents', 'Education', 'Self_Employed',
    'LogIncome', 'CoapplicantIncome_MAD', 'Loan_Amount_Term',
    'Credit_History', 'Loan_Status', 'TotalIncome_MAD',
    'IncomePerTerm', 'Property_Area_Semiurban', 'Property_Area_Urban'
]
TARGET = 'LogLoanAmount'   # On prédit log(montant) pour normaliser la distribution

X = df_model[FEATURES]
y = df_model[TARGET]

# ─── Split Train / Test ───
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ─── Standardisation ───
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'✅ Train : {X_train.shape} | Test : {X_test.shape}')

## 6. 🤖 Entraînement des Modèles

In [ ]:
# ─── Régression Linéaire (baseline) ───
lr = LinearRegression()
lr.fit(X_train_sc, y_train)
print('✅ Régression Linéaire entraînée')

In [ ]:
# ─── Ridge avec validation croisée pour trouver le meilleur alpha ───
alphas_range = np.logspace(-3, 4, 100)

ridge_cv = RidgeCV(alphas=alphas_range, cv=10, scoring='neg_mean_squared_error')
ridge_cv.fit(X_train_sc, y_train)
best_alpha_ridge = ridge_cv.alpha_

print(f'✅ Ridge — Meilleur alpha (CV) : {best_alpha_ridge:.4f}')

ridge = Ridge(alpha=best_alpha_ridge)
ridge.fit(X_train_sc, y_train)

In [ ]:
# ─── Lasso avec validation croisée ───
lasso_cv = LassoCV(alphas=alphas_range, cv=10, max_iter=10000, random_state=42)
lasso_cv.fit(X_train_sc, y_train)
best_alpha_lasso = lasso_cv.alpha_

print(f'✅ Lasso — Meilleur alpha (CV) : {best_alpha_lasso:.4f}')

lasso = Lasso(alpha=best_alpha_lasso, max_iter=10000)
lasso.fit(X_train_sc, y_train)

In [ ]:
# ─── ElasticNet ───
from sklearn.model_selection import GridSearchCV

elasticnet_params = {
    'alpha': [0.001, 0.01, 0.1, 1.0, 10.0],
    'l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9]
}
enet_search = GridSearchCV(
    ElasticNet(max_iter=10000, random_state=42),
    elasticnet_params, cv=10, scoring='neg_mean_squared_error'
)
enet_search.fit(X_train_sc, y_train)
best_enet = enet_search.best_estimator_
print(f'✅ ElasticNet — Meilleurs params : {enet_search.best_params_}')

## 7. 📈 Évaluation et Comparaison des Modèles

In [ ]:
def evaluate_model(name, model, X_tr, X_te, y_tr, y_te):
    """Calcule les métriques et retourne un dictionnaire de résultats."""
    y_pred_train = model.predict(X_tr)
    y_pred_test  = model.predict(X_te)

    # Reconversion depuis log → valeur réelle (MAD)
    y_te_real    = np.expm1(y_te)
    y_pred_real  = np.expm1(y_pred_test)

    return {
        'Modèle': name,
        'R² Train': round(r2_score(y_tr, y_pred_train), 4),
        'R² Test':  round(r2_score(y_te, y_pred_test), 4),
        'RMSE (MAD)': round(np.sqrt(mean_squared_error(y_te_real, y_pred_real)), 0),
        'MAE (MAD)':  round(mean_absolute_error(y_te_real, y_pred_real), 0),
        'MSE Log':    round(mean_squared_error(y_te, y_pred_test), 6),
    }

results = [
    evaluate_model('Régression Linéaire', lr,       X_train_sc, X_test_sc, y_train, y_test),
    evaluate_model('Ridge',               ridge,    X_train_sc, X_test_sc, y_train, y_test),
    evaluate_model('Lasso',               lasso,    X_train_sc, X_test_sc, y_train, y_test),
    evaluate_model('ElasticNet',          best_enet,X_train_sc, X_test_sc, y_train, y_test),
]

results_df = pd.DataFrame(results)
print('=== TABLEAU COMPARATIF DES MODÈLES ===')
results_df.style.highlight_max(subset=['R² Train', 'R² Test'], color='lightgreen') \
               .highlight_min(subset=['RMSE (MAD)', 'MAE (MAD)'], color='lightblue')

In [ ]:
# ─── Graphique comparatif des modèles ───
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

models = results_df['Modèle']
x = np.arange(len(models))
width = 0.35

axes[0].bar(x - width/2, results_df['R² Train'], width, label='R² Train', color='steelblue', alpha=0.85)
axes[0].bar(x + width/2, results_df['R² Test'],  width, label='R² Test',  color='coral', alpha=0.85)
axes[0].set_xlabel('Modèle')
axes[0].set_ylabel('Score R²')
axes[0].set_title('R² Train vs Test par Modèle', fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(models, rotation=15)
axes[0].legend()
axes[0].set_ylim(0, 1)
for bar in axes[0].patches:
    h = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2, h + 0.01, f'{h:.3f}',
                 ha='center', fontsize=9)

axes[1].bar(x, results_df['RMSE (MAD)'], color=['#3498db','#2ecc71','#e74c3c','#9b59b6'], alpha=0.85)
axes[1].set_xlabel('Modèle')
axes[1].set_ylabel('RMSE (Dirhams)')
axes[1].set_title('RMSE en Dirhams par Modèle', fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels(models, rotation=15)
for bar in axes[1].patches:
    h = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2, h + 50, f'{h:,.0f} MAD',
                 ha='center', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# ─── Prédictions vs Valeurs Réelles (meilleur modèle) ───
# Sélectionner le modèle avec le R² Test le plus élevé
best_idx = results_df['R² Test'].idxmax()
best_name = results_df.loc[best_idx, 'Modèle']
best_models = {'Régression Linéaire': lr, 'Ridge': ridge, 'Lasso': lasso, 'ElasticNet': best_enet}
best_model = best_models[best_name]

y_pred_best = np.expm1(best_model.predict(X_test_sc))
y_test_real = np.expm1(y_test)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Scatter réel vs prédit
axes[0].scatter(y_test_real, y_pred_best, alpha=0.5, color='steelblue', s=30)
min_val, max_val = min(y_test_real.min(), y_pred_best.min()), max(y_test_real.max(), y_pred_best.max())
axes[0].plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Parfaite prédiction')
axes[0].set_xlabel('Montant Réel (MAD)')
axes[0].set_ylabel('Montant Prédit (MAD)')
axes[0].set_title(f'{best_name} — Réel vs Prédit', fontweight='bold')
axes[0].legend()

# Résidus
residuals = y_test_real - y_pred_best
axes[1].scatter(y_pred_best, residuals, alpha=0.5, color='coral', s=30)
axes[1].axhline(0, color='black', lw=2, linestyle='--')
axes[1].set_xlabel('Montant Prédit (MAD)')
axes[1].set_ylabel('Résidus (MAD)')
axes[1].set_title(f'{best_name} — Analyse des Résidus', fontweight='bold')

plt.tight_layout()
plt.show()
print(f'\n🏆 Meilleur modèle : {best_name}')

## 8. 🔍 Interprétation des Coefficients

In [ ]:
# ─── Comparaison des coefficients Ridge vs Lasso ───
coef_df = pd.DataFrame({
    'Feature': FEATURES,
    'Ridge': ridge.coef_,
    'Lasso': lasso.coef_,
    'ElasticNet': best_enet.coef_,
})
coef_df['|Ridge|'] = np.abs(coef_df['Ridge'])
coef_df = coef_df.sort_values('|Ridge|', ascending=False)

print('=== COEFFICIENTS DES MODÈLES (triés par importance Ridge) ===')
print(coef_df[['Feature', 'Ridge', 'Lasso', 'ElasticNet']].to_string(index=False))

In [ ]:
# ─── Visualisation des coefficients Ridge ───
fig, axes = plt.subplots(1, 3, figsize=(20, 7))

colors_ridge = ['#e74c3c' if c > 0 else '#3498db' for c in coef_df['Ridge']]
colors_lasso = ['#e74c3c' if c > 0 else '#3498db' for c in coef_df['Lasso']]
colors_enet  = ['#e74c3c' if c > 0 else '#3498db' for c in coef_df['ElasticNet']]

for ax, model_name, col, cols in [
    (axes[0], f'Ridge (α={best_alpha_ridge:.3f})', 'Ridge', colors_ridge),
    (axes[1], f'Lasso (α={best_alpha_lasso:.4f})', 'Lasso', colors_lasso),
    (axes[2], 'ElasticNet', 'ElasticNet', colors_enet)
]:
    bars = ax.barh(coef_df['Feature'], coef_df[col], color=cols, edgecolor='white', alpha=0.85)
    ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
    ax.set_title(model_name, fontsize=12, fontweight='bold')
    ax.set_xlabel('Valeur du coefficient')
    for bar in bars:
        w = bar.get_width()
        if abs(w) > 0.001:
            ax.text(w + (0.002 if w >= 0 else -0.002),
                    bar.get_y() + bar.get_height()/2,
                    f'{w:.3f}', va='center', ha='left' if w >= 0 else 'right', fontsize=8)

plt.suptitle('Comparaison des Coefficients — Ridge vs Lasso vs ElasticNet\n(rouge = positif | bleu = négatif)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ─── Analyse du chemin de régularisation (Ridge) ───
from sklearn.linear_model import ridge_regression

alpha_path = np.logspace(-2, 4, 200)
coefs_path = []

for a in alpha_path:
    r = Ridge(alpha=a)
    r.fit(X_train_sc, y_train)
    coefs_path.append(r.coef_)

coefs_path = np.array(coefs_path)

plt.figure(figsize=(14, 7))
for i, feat in enumerate(FEATURES):
    plt.plot(np.log10(alpha_path), coefs_path[:, i], label=feat, linewidth=2)

plt.axvline(np.log10(best_alpha_ridge), color='black', linestyle='--', linewidth=2,
            label=f'α optimal = {best_alpha_ridge:.3f}')
plt.xlabel('log₁₀(Alpha)', fontsize=13)
plt.ylabel('Coefficient', fontsize=13)
plt.title('Chemin de Régularisation Ridge — Évolution des Coefficients', fontsize=14, fontweight='bold')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# ─── Chemin Lasso : variables sélectionnées ───
from sklearn.linear_model import lasso_path

alphas_lasso_path, coefs_lasso_path, _ = lasso_path(X_train_sc, y_train, alphas=alpha_path)

plt.figure(figsize=(14, 7))
for i, feat in enumerate(FEATURES):
    plt.plot(np.log10(alphas_lasso_path), coefs_lasso_path[i, :], label=feat, linewidth=2)

plt.axvline(np.log10(best_alpha_lasso), color='black', linestyle='--', linewidth=2,
            label=f'α optimal = {best_alpha_lasso:.4f}')
plt.xlabel('log₁₀(Alpha)', fontsize=13)
plt.ylabel('Coefficient', fontsize=13)
plt.title('Chemin de Régularisation Lasso — Sélection de Variables', fontsize=14, fontweight='bold')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
plt.tight_layout()
plt.show()

# Variables nullifiées par Lasso
zero_coefs = coef_df[coef_df['Lasso'] == 0.0]['Feature'].tolist()
print(f'\n🔎 Variables annulées par Lasso (sélection automatique) :')
print(zero_coefs if zero_coefs else 'Aucune — alpha très faible')

## 9. 🧪 Prédiction sur un Nouveau Client

In [ ]:
# ─── Profil d'un nouveau client marocain ───
new_client = {
    'Gender': 1,                    # Homme
    'Married': 1,                   # Marié
    'Dependents': 2,                # 2 personnes à charge
    'Education': 1,                 # Diplômé
    'Self_Employed': 0,             # Salarié
    'LogIncome': np.log1p(15000),   # Revenu total ≈ 15 000 MAD/mois
    'CoapplicantIncome_MAD': 4000,  # Co-demandeur : 4 000 MAD
    'Loan_Amount_Term': 360,        # Durée : 30 ans
    'Credit_History': 1.0,          # Bon historique
    'Loan_Status': 1,               # Pré-approuvé
    'TotalIncome_MAD': 19000,       # Revenu total
    'IncomePerTerm': 19000 / 360,
    'Property_Area_Semiurban': 0,
    'Property_Area_Urban': 1,       # Zone urbaine (ex : Casablanca)
}

client_df = pd.DataFrame([new_client])[FEATURES]
client_scaled = scaler.transform(client_df)

print('\n📋 Profil du client :')
for k, v in new_client.items():
    print(f'   {k:30s} : {v}')

print('\n💰 Prédictions du montant de crédit accordé (en Dirhams) :')
print('─' * 55)
for name, model in [('Régression Linéaire', lr), ('Ridge', ridge),
                     ('Lasso', lasso), ('ElasticNet', best_enet)]:
    pred_log = model.predict(client_scaled)[0]
    pred_mad = np.expm1(pred_log)
    print(f'   {name:25s} : {pred_mad:>12,.0f} MAD')
print('─' * 55)

In [ ]:
# ─── Résumé final ───
print('='*60)
print('📊 RÉSUMÉ FINAL — CRÉDIT SCORING & RÉGRESSION')
print('='*60)
print(f"\n🗂️  Dataset    : {len(df)} observations, {len(FEATURES)} features")
print(f"🎯 Cible      : LoanAmount (en Dirhams MAD)")
print(f"🔄 Split      : 80% train / 20% test")
print()
print(results_df[['Modèle','R² Test','RMSE (MAD)','MAE (MAD)']].to_string(index=False))
print()
print(f"🏆 Meilleur modèle : {results_df.loc[results_df['R² Test'].idxmax(), 'Modèle']}")
print(f"   → R² Test  = {results_df['R² Test'].max():.4f}")
print(f"   → RMSE     = {results_df.loc[results_df['R² Test'].idxmax(), 'RMSE (MAD)']:,.0f} MAD")
print()
print('✅ Analyse terminée avec succès !')